<div align="center"><h1>Accelerating GPT-2 with GPUs: A Deep Dive</h1></div>


![CUDA](https://d29g4g2dyqv443.cloudfront.net/sites/default/files/pictures/2019/NVIDIACuda_Logo.jpg)

## 🚀 Intro

Welcome to this notebook! We're about to explore the fascinating intersection of **Large Language Models (LLMs)** like GPT-2 and the power of **Graphics Processing Units (GPUs)**. Training and running these sophisticated models can be computationally intensive. GPUs, with their massively parallel architecture, offer a significant speed-up, making it feasible to work with models that would otherwise be too slow on traditional CPUs.

This notebook will guide you through understanding how GPU acceleration is leveraged, specifically within the context of a C-based LLM implementation, drawing inspiration from projects like `llm.c`. We'll look at the core concepts and practical aspects of harnessing GPU power for faster training and inference.

---
## 🛠️ Prerequisites

Before we begin, it's helpful to have a basic understanding of:

* **C Programming**: Familiarity with C syntax and concepts will be beneficial as we'll be discussing a C-based implementation.
* **Neural Networks & LLMs**: A foundational knowledge of how neural networks and specifically Large Language Models like GPT-2 operate.
* **GPU Computing (CUDA/OpenCL)**: A conceptual understanding of what GPUs are and how they are used for general-purpose computing. Specific knowledge of CUDA or OpenCL is a plus but not strictly required to follow the high-level concepts.
* **Command Line/Terminal**: Basic comfort with using the command line for running commands and scripts.

## 🎯 Objectives

By the end of this notebook, you should be able to:

* Understand the **role of GPUs** in accelerating LLM computations.
* Identify key **CUDA kernels and techniques** used for optimizing different parts of the GPT-2 model (e.g., attention mechanisms, matrix multiplications).
* Appreciate the **performance benefits** of GPU acceleration compared to CPU-only execution.
* Gain insights into the **challenges and considerations** when implementing and optimizing LLMs on GPUs.
* Have a clearer picture of how projects like `llm.c` aim to provide a simple, hackable, and educational C/CUDA based environment for LLM training.


---
## Accelerated Systems

*Accelerated systems*, also referred to as *heterogeneous systems*, are those composed of both CPUs and GPUs. Accelerated systems run CPU programs which in turn, launch functions that will benefit from the massive parallelism provided by GPUs. This lab environment is an accelerated system which includes an NVIDIA GPU. Information about this GPU can be queried with the `nvidia-smi` (*Systems Management Interface*) command line command. Issue the `nvidia-smi` command now, by `CTRL` + `ENTER` on the code execution cell below. You will find these cells throughout this lab any time you need to execute code. The output from running the command will be printed just below the code execution cell after the code runs. After running the code execution block immediately below, take care to find and note the name of the GPU in the output.

In [1]:
!nvidia-smi           # Verifica el driver y la versión de CUDA soportada
!nvcc --version       # Verifica la versión del compilador CUDA


Sun Sep 27 23:59:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!wget -q https://nvidia.com
!apt-get update -y -qq
!apt-get install -y cuda-nsight-systems-12-8 || apt-get install -y cuda-nsight-systems-12-5 || apt-get install -y nsight-systems-12-6
!apt-get install -y cuda-nsight-systems-12-6 nsight-systems-cli
!ln -sf /usr/local/cuda/bin/nsys /usr/bin/nsys
!ln -sf /opt/nvidia/nsight-systems/*/target-linux-x64/nsys /usr/bin/nsys
!nsys --version

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libxcb-cursor0 libxcb-icccm4 libxcb-image0 libxcb-keysyms1
  libxcb-render-util0 libxcb-util1 libxcb-xinerama0 libxcb-xinput0 libxcb-xkb1
  libxcomposite1 libxkbcommon-x11-0 libxtst6 nsight-systems-2024.6.2
The following NEW packages will be installed:
  cuda-nsight-systems-12-8 libxcb-cursor0 libxcb-icccm4 libxcb-image0
  libxcb-keysyms1 libxcb-render-util0 libxcb-util1 libxcb-xinerama0
  libxcb-xinput0 libxcb-xkb1 libxcomposite1 libxkbcommon

---
## GPT-2 and GPU Acceleration

The following slides present upcoming material visually, at a high level. Click through the slides before moving on to more detailed coverage of their topics in following sections.

In [3]:
%%HTML

<div align="center"><iframe src="https://view.officeapps.live.com/op/view.aspx?src=https://github.com/DennysMallqui/HPC-EP_LLM_PUB/raw/refs/heads/main/Accelerating%20GPT-2%20with%20GPUs%20-%20A%20Deep%20Dive.pptx" width="800px" height="500px" frameborder="0"></iframe></div>

---
## Training GPT-2 Model with Minimal dataset (Romeo and Juliet)

Here we apply training with minimal amount of data to generate a tiny model to test capacity of use GPU in matrix calculations

### Install Dependencies

In [4]:
!yes | pip install -qqq tqdm tiktoken requests datasets

In [5]:
!sudo apt install openmpi-bin openmpi-doc libopenmpi-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openmpi-bin is already the newest version (4.1.6-7ubuntu2).
openmpi-bin set to manually installed.
libopenmpi-dev is already the newest version (4.1.6-7ubuntu2).
libopenmpi-dev set to manually installed.
The following NEW packages will be installed:
  openmpi-doc
0 upgraded, 1 newly installed, 0 to remove and 122 not upgraded.
Need to get 894 kB of archives.
After this operation, 3,404 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 openmpi-doc all 4.1.6-7ubuntu2 [894 kB]
Fetched 894 kB in 2s (455 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requ

### Clone LLM.c project (Reference: https://github.com/karpathy/llm.c.git)

In [6]:
!git clone https://github.com/karpathy/llm.c.git

Cloning into 'llm.c'...
remote: Enumerating objects: 6149, done.
remote: Total 6149 (delta 0), reused 0 (delta 0), pack-reused 6149 (from 1)
Receiving objects: 100% (6149/6149), 2.25 MiB | 17.73 MiB/s, done.
Resolving deltas: 100% (3963/3963), done.


### Install Cudnn dependencies inner LLM.c project

In [7]:
!cd llm.c; git clone https://github.com/NVIDIA/cudnn-frontend.git

Cloning into 'cudnn-frontend'...
remote: Enumerating objects: 29061, done.
remote: Counting objects: 100% (1300/1300), done.
remote: Compressing objects: 100% (675/675), done.
remote: Total 29061 (delta 888), reused 657 (delta 625), pack-reused 27761 (from 3)
Receiving objects: 100% (29061/29061), 215.24 MiB | 18.78 MiB/s, done.
Resolving deltas: 100% (17158/17158), done.


### Make and compile

In [8]:
!cd llm.c; make clean

---------------------------------------------
→ cuDNN is manually disabled by default, run make with `USE_CUDNN=1` to try to enable
✓ OpenMP found
✓ NCCL found, OK to train with multiple GPUs
✓ MPI enabled
✓ nvcc found, including GPU/CUDA support
---------------------------------------------
rm -f train_gpt2 test_gpt2 train_gpt2cu test_gpt2cu train_gpt2fp32cu test_gpt2fp32cu 
rm -f build/*.o


In [9]:
!cd llm.c; export GPU_COMPUTE_CAPABILITY=75; chmod u+x ./dev/download_starter_pack.sh; ./dev/download_starter_pack.sh; make train_gpt2fp32cu

Downloaded tiny_shakespeare_train.bin to /content/llm.c/dev/data/tinyshakespeare/tiny_shakespeare_train.bin
Downloaded tiny_shakespeare_val.bin to /content/llm.c/dev/data/tinyshakespeare/tiny_shakespeare_val.bin
Downloaded gpt2_tokenizer.bin to /content/llm.c/dev/../gpt2_tokenizer.bin
Downloaded gpt2_124M_bf16.bin to /content/llm.c/dev/../gpt2_124M_bf16.bin
Downloaded gpt2_124M_debug_state.bin to /content/llm.c/dev/../gpt2_124M_debug_state.bin
Downloaded gpt2_124M.bin to /content/llm.c/dev/../gpt2_124M.bin
Downloaded hellaswag_val.bin to /content/llm.c/dev/data/hellaswag/hellaswag_val.bin
All files downloaded and saved in their respective directories
---------------------------------------------
→ cuDNN is manually disabled by default, run make with `USE_CUDNN=1` to try to enable
✓ OpenMP found
✓ NCCL found, OK to train with multiple GPUs
✓ MPI enabled
✓ nvcc found, including GPU/CUDA support
---------------------------------------------
/usr/local/cuda/bin/nvcc --threads=0 -t=0 --use_

### Train Process

In [10]:
!cd llm.c; ./train_gpt2fp32cu

+-----------------------+----------------------------------------------------+
| Parameter             | Value                                              |
+-----------------------+----------------------------------------------------+
| train data pattern    | dev/data/tinyshakespeare/tiny_shakespeare_train.bin |
| val data pattern      | dev/data/tinyshakespeare/tiny_shakespeare_val.bin  |
| output log file       | NULL                                               |
| batch size B          | 4                                                  |
| sequence length T     | 1024                                               |
| learning rate         | 0.000300                                           |
| val_loss_every        | 20                                                 |
| val_max_steps         | 20                                                 |
| sample_every          | 20                                                 |
| genT                  | 64                       

### Test from benchmark

In [11]:
!cd llm.c;  export GPU_COMPUTE_CAPABILITY=75; make test_gpt2fp32cu PRECISION=FP32 && ./test_gpt2fp32cu

---------------------------------------------
→ cuDNN is manually disabled by default, run make with `USE_CUDNN=1` to try to enable
✓ OpenMP found
✓ NCCL found, OK to train with multiple GPUs
✓ MPI enabled
✓ nvcc found, including GPU/CUDA support
---------------------------------------------
/usr/local/cuda/bin/nvcc --threads=0 -t=0 --use_fast_math -std=c++17 -O3 --generate-code arch=compute_75,code=[compute_75,sm_75] -DMULTI_GPU -DUSE_MPI test_gpt2_fp32.cu -lcublas -lcublasLt -lnvidia-ml -L/usr/lib/x86_64-linux-gnu/openmpi/lib/ -I/usr/lib/x86_64-linux-gnu/openmpi/include/ -lnccl -lmpi -o test_gpt2fp32cu
[System]
Device 0: Tesla T4
enable_tf32: 0
[State]
batch_size: 4
seq_len: 64
allocated 221 MiB for activations
-43.431618, -43.431652
-39.836346, -39.836403
-43.065910, -43.065929
-42.828045, -42.828091
-43.529541, -43.529575
-44.318398, -44.318436
-41.227425, -41.227444
-41.270760, -41.270771
-42.541393, -42.541443
-42.394997, -42.395069
OK (LOGITS)
allocated 474 MiB for parameter gra

### Generate interfaz

In [12]:
%%writefile llm.c/exec_gpt2_fp32.cu
#define TESTING
#include "train_gpt2_fp32.cu" // Include all GPU setup + GPT2 structs and functions

#define MAX_GEN_TOKENS 50

// Tokenizer structure and functions as defined in the provided code
#include <stdint.h>
#include <ctype.h>
#include <assert.h>

int tokenizer_encode(Tokenizer *tokenizer,
                     const char *text,
                     int *input_ids,
                     int max_len)
{
    if (!tokenizer || !tokenizer->init_ok || !text || !input_ids) {
        return 0;
    }

    int text_len = (int)strlen(text);
    int pos = 0;
    int count = 0;

    while (pos < text_len && count < max_len) {
        int best_id = -1;
        int best_len = 0;

        // 1) Try to match the longest token in vocab
        for (uint32_t i = 0; i < tokenizer->vocab_size; ++i) {
            char *tok = tokenizer->token_table[i];
            int tok_len = (int)strlen(tok);
            if (tok_len == 0 || tok_len > text_len - pos) continue;
            if (strncmp(text + pos, tok, tok_len) == 0) {
                if (tok_len > best_len) {
                    best_len = tok_len;
                    best_id  = (int)i;
                }
            }
        }

        // 2) Fallback: single-byte token
        if (best_id < 0) {
            unsigned char c = (unsigned char)text[pos];
            for (uint32_t i = 0; i < tokenizer->vocab_size; ++i) {
                // token_table[i] is a C-string, so check length==1 and same byte
                if (tokenizer->token_table[i][0] == (char)c
                 && tokenizer->token_table[i][1] == '\0') {
                    best_id = (int)i;
                    best_len = 1;
                    break;
                }
            }
        }

        // 3) If still no match, skip this byte
        if (best_id < 0) {
            pos += 1;
            continue;
        }

        input_ids[count++] = best_id;
        pos += best_len;
    }

    return count;
}

// Main function
int main(int argc, char *argv[]) {
    // set up the device
    int deviceIdx = 0;
    cudaCheck(cudaSetDevice(deviceIdx));
    cudaDeviceProp deviceProp;
    cudaGetDeviceProperties(&deviceProp, deviceIdx);
    printf("[System]\n");
    printf("Device %d: %s\n", deviceIdx, deviceProp.name);

    // setup cuBLAS and cuBLASLt
    cublasCheck(cublasCreate(&cublas_handle));
    // TF32 precision is equivalent to torch.set_float32_matmul_precision('high')
    int enable_tf32 = deviceProp.major >= 8 ? 1 : 0;
    enable_tf32 = 0; // NOTE: disable TF32 for testing!!!
    printf("enable_tf32: %d\n", enable_tf32);
    cublas_compute_type = enable_tf32 ? CUBLAS_COMPUTE_32F_FAST_TF32 : CUBLAS_COMPUTE_32F;
    cublasMath_t cublas_math_mode = enable_tf32 ? CUBLAS_TF32_TENSOR_OP_MATH : CUBLAS_DEFAULT_MATH;
    cublasCheck(cublasSetMathMode(cublas_handle, cublas_math_mode));

    // build the GPT-2 model from a checkpoint
    GPT2 model;
    gpt2_build_from_checkpoint(&model, "gpt2_124M.bin");

    // int C = model.config.channels;
    int V = model.config.vocab_size;
    int Vp = model.config.padded_vocab_size;
    int maxT = model.config.max_seq_len;
    // int L = model.config.num_layers;

    // load additional information that we will use for debugging and error checking
    FILE *state_file = fopenCheck("gpt2_124M_debug_state.bin", "rb");
    int state_header[256];
    freadCheck(state_header, sizeof(int), 256, state_file);
    if (state_header[0] != 20240327) { printf("Bad magic state file\n"); exit(EXIT_FAILURE); }
    if (state_header[1] != 2) {
        fprintf(stderr, "Bad version in state file\n");
        fprintf(stderr, "---> HINT: try to re-run `python train_gpt2.py`\n");
        exit(EXIT_FAILURE);
    }
    int B = state_header[2]; // batch size, e.g. 4
    int T = state_header[3]; // time / sequence length (e.g. 64, up to maxT)
    assert(0 <= T && T <= maxT);
    printf("[State]\n");
    printf("batch_size: %d\n", B);
    printf("seq_len: %d\n", T);

    ParameterTensors expected_grads; // will be read from file (from PyTorch)
    ParameterTensors calculated_grads; // will be calculated by us
    float* expected_grads_memory = malloc_and_point_parameters(&expected_grads, model.param_sizes, 0);
    float* calculated_grads_memory = malloc_and_point_parameters(&calculated_grads, model.param_sizes, 0);

    // inputs and expected outputs, only used for error checking
    int* x = (int*)mallocCheck(B * T * sizeof(int));
    int* y = (int*)mallocCheck(B * T * sizeof(int));
    float* expected_logits = (float*) mallocCheck(B * T * V * sizeof(float));
    float* expected_loss = (float*) mallocCheck(1 * sizeof(float));

    // read reference information from Python
    freadCheck(x, sizeof(int), B*T, state_file);
    freadCheck(y, sizeof(int), B*T, state_file);
    freadCheck(expected_logits, sizeof(float), B*T*V, state_file);
    freadCheck(expected_loss, sizeof(float), 1, state_file);
    freadCheck(expected_grads_memory, sizeof(float), model.num_parameters, state_file);
    fcloseCheck(state_file);

    // Load tokenizer vocab
    Tokenizer tokenizer;
    tokenizer_init(&tokenizer, "gpt2_tokenizer.bin");

    // Input prompt
    const char *prompt = argc >= 2 ? argv[1] : "Romeo and Juliet are";
    printf("[Prompt] %s\n", prompt);

    // Tokenize input
    int input_ids[model.config.max_seq_len] = {0};
    int input_len = tokenizer_encode(&tokenizer, prompt, input_ids, model.config.max_seq_len);
    if (input_len == 0) {
        fprintf(stderr, "Failed to tokenize input.\n");
        return 1;
    }

    //printf("[input_len] %d\n", input_len);

    // Generate tokens (greedy decoding)
    int cur_len = input_len;
    for (int i = 0; i < MAX_GEN_TOKENS && cur_len < model.config.max_seq_len; ++i) {
        //printf("[loop token] %d\n", i);
        gpt2_forward(&model, input_ids, NULL, B, T);

        //printf("[cur_len] %d\n", cur_len);

        // Get last token logits
        float *logits = (float *)malloc(sizeof(float) * model.config.padded_vocab_size);
        cudaMemcpy(logits, model.acts.output + (cur_len - 1) * model.config.padded_vocab_size,
                   sizeof(float) * model.config.padded_vocab_size, cudaMemcpyDeviceToHost);

        // Find the token with highest probability (greedy)
        int max_id = 0;
        float max_val = logits[0];
        for (int j = 1; j < model.config.vocab_size; j++) {
            if (logits[j] > max_val) {
                max_val = logits[j];
                max_id = j;
            }
        }
        free(logits);

        input_ids[cur_len] = max_id;
        cur_len++;

        if (max_id == tokenizer.eot_token) {
            break;
        }
    }

    // Decode the output
    char decoded[2048];
    decoded[0] = '\0';  // Clear the decoded string
    for (int i = input_len; i < cur_len; ++i) {
        if (i > 0) {
            strncat(decoded, " ", sizeof(decoded) - strlen(decoded) - 1);
        }
        const char *token = tokenizer_decode(&tokenizer, input_ids[i]);
        if (token) {
            strncat(decoded, token, sizeof(decoded) - strlen(decoded) - 1);
        }
        if (strcmp(token, ".") == 0) {
            break;
        }
    }

    printf("[Output]\n%s\n", decoded);

    // Free tokenizer memory
    tokenizer_free(&tokenizer);

    return 0;
}


Writing llm.c/exec_gpt2_fp32.cu


In [13]:
!cd llm.c; /usr/local/cuda/bin/nvcc --threads=0 -t=0 --use_fast_math -std=c++17 -O3 --generate-code arch=compute_75,code=[compute_75,sm_75] -DMULTI_GPU -DUSE_MPI exec_gpt2_fp32.cu -lcublas -lcublasLt -lnvidia-ml -L/usr/lib/x86_64-linux-gnu/openmpi/lib/ -I/usr/lib/x86_64-linux-gnu/openmpi/include/ -lnccl -lmpi -o exec_gpt2fp32cu

exec_gpt2_fp32.cu(94): warning #177-D: variable "Vp" was declared but never referenced
      int Vp = model.config.padded_vocab_size;
          ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"



### Finally Test inference!

In [14]:
!cd llm.c; ./exec_gpt2fp32cu "Romeo and Juliet are "

[System]
Device 0: Tesla T4
enable_tf32: 0
[State]
batch_size: 4
seq_len: 64
[Prompt] Romeo and Juliet are 
allocated 221 MiB for activations
[Output]
   the  only  two  characters  who  are  not  in  the  same  class .


In [15]:
!apt-get install nsight-systems

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaccinj64-12.0 libcuinj64-12.0 libcupti12 libibumad3 libjpeg62
  libnvidia-cfg1-580 libnvidia-compute-535 libnvidia-compute-580
  libnvidia-decode-580 libnvidia-gpucomp-580 libpfm4 node-html5shiv
  nsight-systems-target nvidia-persistenced nvidia-profiler
Suggested packages:
  nvidia-driver-pinning-580 nodejs nvidia-cuda-toolkit
The following NEW packages will be installed:
  libaccinj64-12.0 libcuinj64-12.0 libcupti12 libibumad3 libjpeg62
  libnvidia-cfg1-580 libnvidia-compute-535 libnvidia-compute-580
  libnvidia-decode-580 libnvidia-gpucomp-580 libpfm4 node-html5shiv
  nsight-systems nsight-systems-target nvidia-persistenced nvidia-profiler
0 upgraded, 16 newly installed, 0 to remove and 122 not upgraded.
Need to get 270 MB of archives.
After this operation, 1,132 MB of additional disk space will be used.
Get:1 http://archive.ubunt

In [16]:
!cd llm.c && nsys profile --stats=true ./train_gpt2fp32cu



+-----------------------+----------------------------------------------------+
| Parameter             | Value                                              |
+-----------------------+----------------------------------------------------+
| train data pattern    | dev/data/tinyshakespeare/tiny_shakespeare_train.bin |
| val data pattern      | dev/data/tinyshakespeare/tiny_shakespeare_val.bin  |
| output log file       | NULL                                               |
| batch size B          | 4                                                  |
| sequence length T     | 1024                                               |
| learning rate         | 0.000300                                           |
| val_loss_every        | 20                                                 |
| val_max_steps         | 20                                                 |
| sample_every          | 20                                                 |
| genT                  | 64                       